# Chroma Vector Store with LangChain + Gemini

Converted to use `langchain_google_genai` (Gemini) for embeddings, `.env` for API keys, and the `langchain-chroma` package (required for LangChain v1.x, since `Chroma` was removed from `langchain.vectorstores`).

In [1]:
# install dependencies
# langchain v1.x splits Chroma into its own package: langchain-chroma
!pip install langchain langchain-google-genai langchain-chroma python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

In [4]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [5]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [7]:
# add documents
vector_store.add_documents(docs)

['3e4c8da9-51c8-4eb4-b662-f161e25f061d',
 '28b12cb6-5f89-4e87-9bf2-0944b7f9cf26',
 '6d52dfd3-1950-4201-97fa-eb059ca46ec6',
 '4f8e0971-856b-494d-8930-7b49bfa0a811',
 'c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22']

In [8]:
# view documents
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['3e4c8da9-51c8-4eb4-b662-f161e25f061d',
  '28b12cb6-5f89-4e87-9bf2-0944b7f9cf26',
  '6d52dfd3-1950-4201-97fa-eb059ca46ec6',
  '4f8e0971-856b-494d-8930-7b49bfa0a811',
  'c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [9]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='4f8e0971-856b-494d-8930-7b49bfa0a811', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [10]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='4f8e0971-856b-494d-8930-7b49bfa0a811', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(id='c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

In [12]:
# meta-data filtering + similarity
vector_store.similarity_search_with_score(
    query="best player",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='6d52dfd3-1950-4201-97fa-eb059ca46ec6', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.7040836811065674),
 (Document(id='c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.7358927726745605)]

In [13]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id=vector_store.get()['ids'][0], document=updated_doc1)

In [14]:
# view documents
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['3e4c8da9-51c8-4eb4-b662-f161e25f061d',
  '28b12cb6-5f89-4e87-9bf2-0944b7f9cf26',
  '6d52dfd3-1950-4201-97fa-eb059ca46ec6',
  '4f8e0971-856b-494d-8930-7b49bfa0a811',
  'c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple c

In [15]:
# delete document
vector_store.delete(ids=[vector_store.get()['ids'][0]])

In [16]:
# view documents
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['28b12cb6-5f89-4e87-9bf2-0944b7f9cf26',
  '6d52dfd3-1950-4201-97fa-eb059ca46ec6',
  '4f8e0971-856b-494d-8930-7b49bfa0a811',
  'c4c2a7eb-7c9d-436d-8fd7-d4e568d54b22'],
 'embeddings': array([[-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(4, 3072)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah i